# HAR Stage 10 — GSE270828 replicate-matched pseudobulk RNA counts (no Google Drive)

**Goal:** convert original GSE270828 RDS's *verified RNA raw-count matrix* into a compact, reproducible **gene × (replicate × HAR)** aggregated count matrix. The aggregated counts allow us to design appropriate statistical tests; this notebook **does not itself compute differential expression**, P values, or validated gene targets.

**Important safeguards**
- Uses `single_true` guide assignments; excludes `single_biased` as a predefined sensitivity decision.
- Uses exactly the published three `rep` labels, not the 24 lanes or the individual cells as independent experimental units. Independent biological donor status is **not established**.
- Stops if raw RNA counts, gene names, or cell-barcode alignment cannot be verified.
- Does not use or request Google Drive; it reuses the file in the current Colab runtime or downloads it from original GEO (2.31 GB).
- On success exports **Pseudobulk_raw_counts.npz**, **Pseudobulk_Group_Metadata.tsv**, **Pseudobulk_Gene_IDs.tsv**, and logs in a results ZIP.
- If the Colab runtime has limited RAM, the program may fail safely. The final ZIP then contains diagnostic logs rather than fabricated results.

**Instructions:** Runtime → Run all; download `HAR_Stage10_Pseudobulk_Results.zip` and upload it to this chat. If it is unusually large, upload the `.npz` and metadata files separately. This is a *pilot expression measurement*, not a causality analysis.


In [ ]:
import os,sys,time,json,subprocess,zipfile,shutil,traceback,hashlib
from pathlib import Path
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
SOURCE_DIR = ROOT / 'GSE270828_public_source'; SOURCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = ROOT / 'HAR_Stage10_Output'; OUTPUT.mkdir(parents=True, exist_ok=True)
SOURCE = SOURCE_DIR / 'GSE270828_Merged_CountMatrix.rds.gz'
URL='https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/suppl/GSE270828_Merged_CountMatrix.rds.gz'
EXPECTED_SIZE=2312020968
FINAL_ZIP=ROOT/'HAR_Stage10_Pseudobulk_Results.zip'
status={'stage':10,'data_source':'GSE270828 original GEO double gzip RDS','google_drive_used':False,'differential_expression_performed':False,'status':'started','matched_replicates_are_independent_donors':False,'group':'rep x har','assigment_filter':'num_guide == single_true'}
def save_status(): (OUTPUT/'STATUS.json').write_text(json.dumps(status,indent=2,default=str))

# The original data are stored in the temporary Colab workspace, not Google Drive.
# We reuse the 2.31 GB file from earlier stages when this runtime is still alive.
if SOURCE.is_file() and SOURCE.stat().st_size==EXPECTED_SIZE:
    print('Using original file already present:', SOURCE)
    status['download']='reused'
else:
    print('Original source missing; attempting direct NCBI download (2.31 GB).')
    try:
        import requests
        partial=SOURCE.with_suffix(SOURCE.suffix+'.part')
        for attempt in range(1,5):
            offset=partial.stat().st_size if partial.exists() else 0
            headers={'Range':f'bytes={offset}-'} if offset else {}
            try:
                with requests.get(URL,stream=True,headers=headers,timeout=(30,120)) as resp:
                    resp.raise_for_status()
                    use_append=offset>0 and resp.status_code==206
                    with open(partial,'ab' if use_append else 'wb') as dest:
                        for chunk in resp.iter_content(chunk_size=4*1024*1024):
                            if chunk: dest.write(chunk)
                if partial.stat().st_size!=EXPECTED_SIZE: raise RuntimeError(f'Incomplete data: {partial.stat().st_size} bytes, expected {EXPECTED_SIZE}')
                partial.rename(SOURCE);status['download']='downloaded';break
            except Exception as e:
                print(f'Attempt {attempt}: {e}')
                if attempt==4:raise
                time.sleep(5*attempt)
        print('Download complete:',SOURCE.stat().st_size,'bytes')
    except Exception as exc:
        status['download_error']=str(exc);status['status']='download_failed';save_status()
        print('Download failed, diagnostic ZIP will still be created:',exc)
save_status()


In [ ]:
R_CODE = "\nargs <- commandArgs(trailingOnly=TRUE)\nsource_path<-args[[1]]; out<-args[[2]]\ndir.create(out,recursive=TRUE,showWarnings=FALSE)\nlogf<-file.path(out,'R_progress.log')\nlogmsg<-function(...) cat(format(Sys.time(),'%Y-%m-%d %H:%M:%S'),...,'\\n',file=logf,append=TRUE)\nlogmsg('START')\nsuppressPackageStartupMessages(library(Matrix))\n# Previous stage verified the GEO source is a double-gzip stream containing RDS.\ncmd<-paste('gzip -dc',shQuote(normalizePath(source_path)), '| gzip -dc')\ncon<-pipe(cmd,open='rb')\nlogmsg('RDS_DESERIALIZATION_STARTED')\nobj<-readRDS(con)\ntry(close(con),silent=TRUE)\nlogmsg('RDS_LOADED',class(obj)[[1]])\nmd<-attr(obj,'meta.data',exact=TRUE)\nif(!is.data.frame(md)) stop('S4 metadata not found, cannot link count columns to cell metadata')\nrequired<-c('rep','har','guide','num_guide')\nif(!all(required %in% colnames(md))) stop('Missing required columns in metadata')\n\n# Base R S4 attribute lookup avoids a hard dependency on SeuratObject.\nassays<-attr(obj,'assays',exact=TRUE)\nif(is.null(assays) || !length(assays)) stop('Assay object not available: do not fabricate counts')\nlogmsg('ASSAYS',paste(names(assays),collapse=','))\nrna<-assays[['RNA']]\nif(is.null(rna)) stop('RNA assay absent')\ncounts<-attr(rna,'counts',exact=TRUE)\nif(is.null(counts)){\n    layers<-attr(rna,'layers',exact=TRUE)\n    if(is.list(layers) && 'counts' %in% names(layers)) counts<-layers[['counts']]\n}\nif(is.null(counts)) stop('Untransformed RNA count matrix not available; requires supported assay import')\nif(!inherits(counts,'sparseMatrix')) stop('Matrix is not sparse; halted to avoid RAM overload and uncertain structure')\nlogmsg('COUNT_MATRIX',paste(dim(counts),collapse='x'),'NNZ',length(counts@x))\nif(any(!is.finite(counts@x))) stop('Count matrix includes nonfinite entries')\nif(any(counts@x < 0) || any(abs(counts@x-round(counts@x))>1e-6)) stop('RNA matrix not verified as nonnegative integer raw counts')\nif(ncol(counts)!=nrow(md)) stop('RNA column count does not match cell metadata rows')\nif(is.null(colnames(counts)) || is.null(rownames(md))) stop('Cell identifiers absent; stopping, no assumption of ordering')\nif(anyDuplicated(colnames(counts)) || anyDuplicated(rownames(md))) stop('Duplicate cell identifiers')\nif(!setequal(colnames(counts),rownames(md))) stop('RNA count cell identifiers do not match metadata')\nif(!identical(colnames(counts),rownames(md))){\n    logmsg('REORDER_COUNT_COLUMNS')\n    counts<-counts[,match(rownames(md),colnames(counts)),drop=FALSE]\n}\nkeep<-as.character(md$num_guide)=='single_true' & !is.na(md$har) & !is.na(md$rep)\nif(sum(keep)<1000L) stop('Insufficient reliable single_true guide assignments')\ngrp<-paste(as.character(md$rep[keep]),as.character(md$har[keep]),sep='__')\nlevels<-sort(unique(grp))\nidx<-which(keep)\nI<-Matrix::sparseMatrix(i=idx,j=match(grp,levels),x=rep(1,length(idx)),dims=c(ncol(counts),length(levels)))\nlogmsg('PSEUDOBULK_AGGREGATION_BEGIN','groups',length(levels),'cells',length(idx))\nagg<-counts %*% I\nrm(I,counts,obj,rna,assays);gc(verbose=FALSE)\nlogmsg('AGGREGATED',paste(dim(agg),collapse='x'))\nif(is.null(rownames(agg))) stop('Gene identifiers missing in aggregate count matrix')\n# Count columns and metadata are in the same deterministic alphabetical group order.\nmeta<-data.frame(group_id=levels, rep=sub('__.*$','',levels), har=sub('^[^_]+__','',levels), stringsAsFactors=FALSE)\nmeta$n_cells<-as.integer(table(factor(grp,levels=levels)))\nif(!all(meta$n_cells>0L)) stop('Zero-cell pseudobulk group')\nwrite.table(meta,file=file.path(out,'Pseudobulk_Group_Metadata.tsv'),sep='\\t',quote=FALSE,row.names=FALSE)\nwrite.table(data.frame(gene=rownames(agg)),file=file.path(out,'Pseudobulk_Gene_IDs.tsv'),sep='\\t',quote=FALSE,row.names=FALSE)\n# Gene rows x group columns. Raw count matrix, NOT normalized expression.\nMatrix::writeMM(agg,file=file.path(out,'Pseudobulk_raw_counts.mtx'))\nlogmsg('SUCCESS','nnz',length(agg@x),'genes',nrow(agg),'groups',ncol(agg))\n"

In [ ]:
# Extract only raw-count pseudobulk matrices, not individual-cell raw data.
# This may take memory; errors and logs are exported in the final ZIP.
import shutil
from pathlib import Path
script=OUTPUT/'stage10_pseudobulk_baseR.R'
script.write_text(R_CODE)
if not SOURCE.is_file() or SOURCE.stat().st_size != EXPECTED_SIZE:
    status['status']='download_failed';status['error']='Original GEO download absent/incomplete';save_status()
    print('No complete source file: creating diagnostic ZIP in final cell, not analysing expression.')
try:
    if not SOURCE.is_file() or SOURCE.stat().st_size != EXPECTED_SIZE:
        raise RuntimeError('Original GEO download absent/incomplete; cannot load expression')
    if shutil.which('Rscript') is None:
        print('R missing: installing R + Matrix package into the Colab runtime.')
        subprocess.run(['bash','-lc','apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq r-base-core r-cran-matrix'],check=True,timeout=1200)
    result=subprocess.run(['Rscript',str(script),str(SOURCE),str(OUTPUT)],capture_output=True,text=True,timeout=2400)
    (OUTPUT/'R_stdout.log').write_text(result.stdout[-50000:]); (OUTPUT/'R_stderr.log').write_text(result.stderr[-50000:])
    print('R exit code:',result.returncode)
    if result.returncode!=0:raise RuntimeError('R exited with error: '+result.stderr[-1800:])
    mtx=OUTPUT/'Pseudobulk_raw_counts.mtx'
    if not mtx.is_file():raise RuntimeError('Output matrix missing despite R success')
    print('Reading Matrix Market and converting to compressed SciPy sparse NPZ...')
    from scipy.io import mmread
    from scipy import sparse
    mat=mmread(str(mtx)).tocsr()
    if not sparse.issparse(mat) or mat.shape[0]<100 or mat.shape[1]<100:raise RuntimeError('Unexpected pseudobulk matrix dimensions')
    if not (mat.data>=0).all():raise RuntimeError('Negative count detected')
    sparse.save_npz(str(OUTPUT/'Pseudobulk_raw_counts.npz'),mat,compressed=True)
    status['status']='complete';status['genes']=mat.shape[0];status['groups']=mat.shape[1];status['nonzero_counts']=int(mat.nnz)
    status['raw_count_matrix_included']=True
    print('Exported counts:',mat.shape,'NNZ:',mat.nnz)
    mtx.unlink()  # the NPZ preserves all cells; remove redundant large text matrix
except Exception as exc:
    status['status']='failed_with_diagnostics';status['error']=str(exc)
    print('STOP (no invented results):',exc)
save_status()


In [ ]:
# A ZIP is always generated, including errors when the run cannot complete.
with zipfile.ZipFile(FINAL_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTPUT.glob('*')):
        if p.is_file() and p.name!='Pseudobulk_raw_counts.mtx':z.write(p,p.name)
print('Result ZIP:',FINAL_ZIP,'Size MB:',round(FINAL_ZIP.stat().st_size/1e6,2))
if Path('/content').exists():
    from google.colab import files
    files.download(str(FINAL_ZIP))
